In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("raw_path", "/Volumes/workspace/tfl/raw/tfl_cycling_2026/")
dbutils.widgets.text("bronze", "workspace.tfl_lakehouse.bronze")
dbutils.widgets.text("source_file", "source_file")

raw_path = dbutils.widgets.get("raw_path")
bronze = dbutils.widgets.get("bronze")
source_file = dbutils.widgets.get("source_file")

raw_files = [f.path.removeprefix("dbfs:") for f in dbutils.fs.ls(raw_path) if not f.path.endswith("/")]
print(f"Found {len(raw_files)} files in {raw_path}")

raw_df = spark.createDataFrame([(p,) for p in raw_files], "source_file string")

if spark.catalog.tableExists(bronze):
    loaded_df = (
        spark.table(bronze)
        .select(F.col(source_file).alias("source_file"))
        .distinct()
    )
    new_df = raw_df.join(loaded_df, on="source_file", how="left_anti")
else:
    new_df = raw_df

new_files = [r.source_file for r in new_df.collect()]
print(f"{len(new_files)} new files to process")


dbutils.jobs.taskValues.set(key="new_file_count", value=len(new_files))